In [7]:
import requests
import json
import re
import pandas as pd
import numpy as np

def get_uniprot(accession):
    '''
    define http_function to get the data from Uniprot API
    '''
    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"

    http_args = {
        "params": {"format": "json"} 
    }
    def http_function(url, **kwargs):
        response = requests.get(url, **kwargs)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code} ")
            return None
    return http_function(endpoint, **http_args)
    

def uniprot_parse_response(resp: dict):
    '''
    parse response from Uniprot and output
    organism, geneInfo, sequenceInfo, type

    do not forget to include error handling
    '''

    output = {}

    try:
        output["organism"] = resp.get("organism", {}).get("scientificName", None)

        genes = resp.get("genes", [])
        if genes:
            output["geneInfo"] = genes[0]
        else:
            output["geneInfo"] = None

        output["sequenceInfo"] = resp.get("sequence", {})

        output["type"] = "protein"


    except Exception as e:
        print("Error parsing response:", e)
        return None

    return output
    
def get_ensembl(id):
    '''
    define http_function to get the data from Ensembl API
    '''
    endpoint = f'https://rest.ensembl.org/lookup/id/{id}'
    
    http_args = {"headers": {"Content-Type" : "application/json"}}
    def http_function(url, **kwargs):
        response = requests.get(url, **kwargs)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code} ")
            return None
    return http_function(endpoint, **http_args)


def ensembl_parse_response(resp: dict):
    '''
    parse Ensembl response and output
    object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

    do not forget to include error handling (e.g. for key errors)
    '''
    output = {}

    try:
        # organism
        output['object_type'] = resp['object_type']
        output['assembly_name'] = resp['assembly_name']
        output['species'] = resp['species']
        output['db_type'] = resp['db_type']
        output['biotype'] = resp['biotype']
        output['db_type'] = resp['db_type']
        output['display_name'] = resp['display_name']
        output['id'] = resp['id']
        output['description'] = resp['description']
        output['canonical_transcript'] = resp['canonical_transcript']
        output['source'] = resp['source']
    except Exception as e:
        print("Error parsing response:", e)
        return None

    return output



def main(ids: list):
    '''
    Function that iterates over all the provided IDs and parses them into dict,
    transforms into pandas.DataFrame, and return it
    {ID : info from parse_response(), ...}

    If ID is incorrect, it should return {ID : error message}
    '''
    output = {}
    for id in ids:

        try:

            if re.match(r"ENSG[0-9]+", id):
                resp = get_ensembl(id)
                parsed = ensembl_parse_response(resp)
                
            elif re.match(r"[OPQ][0-9][A-Z0-9]{3}[0-9]", id) or re.match(r"[A-NR-Z][0-9]{5}", id):
                resp = get_uniprot(id)
                parsed = uniprot_parse_response(resp)

            else:
                parsed = {"error": "Unknown ID format"}

            output[id] = parsed

        except Exception as e:
            output[id] = {"error": str(e)}
            
    output = pd.DataFrame.from_dict(output, orient="index")

    return output

In [10]:
get_uniprot('P11473')

{'entryType': 'UniProtKB reviewed (Swiss-Prot)',
 'primaryAccession': 'P11473',
 'secondaryAccessions': ['B2R5Q1', 'G3V1V9', 'Q5PSV3'],
 'uniProtkbId': 'VDR_HUMAN',
 'entryAudit': {'firstPublicDate': '1989-10-01',
  'lastAnnotationUpdateDate': '2026-01-28',
  'lastSequenceUpdateDate': '1989-10-01',
  'entryVersion': 265,
  'sequenceVersion': 1},
 'annotationScore': 5.0,
 'organism': {'scientificName': 'Homo sapiens',
  'commonName': 'Human',
  'taxonId': 9606,
  'lineage': ['Eukaryota',
   'Metazoa',
   'Chordata',
   'Craniata',
   'Vertebrata',
   'Euteleostomi',
   'Mammalia',
   'Eutheria',
   'Euarchontoglires',
   'Primates',
   'Haplorrhini',
   'Catarrhini',
   'Hominidae',
   'Homo']},
 'proteinExistence': '1: Evidence at protein level',
 'proteinDescription': {'recommendedName': {'fullName': {'value': 'Vitamin D3 receptor'},
   'shortNames': [{'value': 'VDR'}]},
  'alternativeNames': [{'fullName': {'value': '1,25-dihydroxyvitamin D3 receptor'}},
   {'fullName': {'value': 'Nuc

In [11]:
get_uniprot('helloworld')

Error 400 


In [12]:
get_uniprot('helloworld').json()

Error 400 


<class 'AttributeError'>: 'NoneType' object has no attribute 'json'

In [13]:
uniprot_parse_response(get_uniprot('P11473'))

{'organism': 'Homo sapiens',
 'geneInfo': {'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
     'source': 'HGNC',
     'id': 'HGNC:12679'}],
   'value': 'VDR'},
  'synonyms': [{'value': 'NR1I1'}]},
 'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
  'length': 427,
  'molWeight': 48289,
  'crc64': 'F95F300D042C4CB7',
  'md5': '0D963ACD4A34674368324EE026023597'},
 'type': 'protein'}

In [4]:
uniprot_parse_response(get_uniprot('P11473'))

{'organism': 'Homo sapiens',
 'geneInfo': {'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
     'source': 'HGNC',
     'id': 'HGNC:12679'}],
   'value': 'VDR'},
  'synonyms': [{'value': 'NR1I1'}]},
 'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
  'length': 427,
  'molWeight': 48289,
  'crc64': 'F95F300D042C4CB7',
  'md5': '0D963ACD4A34674368324EE026023597'},
 'type': 'protein'}

In [14]:
get_ensembl('ENSMUSG00000041147')

{'version': 11,
 'strand': 1,
 'species': 'mus_musculus',
 'start': 150446095,
 'assembly_name': 'GRCm39',
 'biotype': 'protein_coding',
 'end': 150493794,
 'id': 'ENSMUSG00000041147',
 'canonical_transcript': 'ENSMUST00000044620.11',
 'db_type': 'core',
 'source': 'ensembl_havana',
 'display_name': 'Brca2',
 'logic_name': 'ensembl_havana_gene_mus_musculus',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'object_type': 'Gene',
 'seq_region_name': '5'}

In [15]:
get_ensembl('helloworld')

Error 400 


In [16]:
get_ensembl('helloworld').json()

Error 400 


<class 'AttributeError'>: 'NoneType' object has no attribute 'json'

In [17]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'object_type': 'Gene',
 'assembly_name': 'GRCm39',
 'species': 'mus_musculus',
 'db_type': 'core',
 'biotype': 'protein_coding',
 'display_name': 'Brca2',
 'id': 'ENSMUSG00000041147',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'canonical_transcript': 'ENSMUST00000044620.11',
 'source': 'ensembl_havana'}

In [9]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

,organism,geneInfo,sequenceInfo,type,error,object_type,assembly_name,species,db_type,biotype,display_name,id,description,canonical_transcript,source
P11473,Homo sapiens,{'geneName': {'evidences': [{'evidenceCode': '...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,{'geneName': {'value': 'INS'}},{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,Unknown ID format,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRAF,ENSG00000157764,"B-Raf proto-oncogene, serine/threonine kinase ...",ENST00000646891.2,ensembl_havana
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,GRCh38,homo_sapiens,core,protein_coding,BRCA2,ENSG00000139618,BRCA2 DNA repair associated [Source:HGNC Symbo...,ENST00000380152.8,ensembl_havana
